In [1]:
import pandas as pd
import joblib
from backtesting import Backtest, Strategy
import pytz # We'll need this for our dynamic session times
from datetime import datetime, time

c:\Users\mecha\anaconda3\envs\trade_env\Lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

### **Step 2: Building the Strategy's Brain**
This is the most important part. We will define a class that contains our trading logic.

In [2]:
# --- Step 2: Define the Trading Strategy ---
print("Defining the ML-based trading strategy...")

class MLStrategy(Strategy):
    # --- The init() method is called once at the start ---
    def init(self):
        print("Initializing strategy...")
        # 1. Load the pre-trained champion model
        model_path = '../models/xgb_classifier_hyp_a_xauusd_h1_2018_present.joblib'
        self.model = joblib.load(model_path)
        print("Model loaded successfully.")

        # 2. Load the feature data that corresponds to the model
        features_path = '../data/processed/hyp_a_features_xauusd_h1_2018_present.parquet'
        self.features = pd.read_parquet(features_path)
        print("Feature data loaded successfully.")
        
        # 3. Store the London timezone for dynamic open times
        self.london_tz = pytz.timezone('Europe/London')

    # --- The next() method is called for each new candle ---
    def next(self):
        # self.data.index[-1] gives us the timestamp of the current candle
        current_time_utc = self.data.index[-1]
        
        # --- Determine the London open time for THIS specific day ---
        current_date = current_time_utc.date()
        london_open_local = self.london_tz.localize(datetime.combine(current_date, time(8, 0)))
        london_open_utc = london_open_local.astimezone(pytz.utc)

        # --- TRADING LOGIC ---
        # We only want to make one decision per day, exactly at the London open.
        if current_time_utc == london_open_utc:
            
            # Defensive check: Make sure we have features for today
            if current_date not in self.features.index:
                return # If no features, do nothing

            # 1. Get today's features
            # We select the row for the current date and drop the target columns
            today_features = self.features.loc[[current_date]].drop(columns=['london_direction', 'london_return', 'timeframe', 'symbol'])
            
            # 2. Use the model to make a prediction
            prediction = self.model.predict(today_features)[0] # [0] to get the single value
            
            # 3. Execute the trade based on the prediction
            # We will also close any existing position before opening a new one.
            # This ensures we only hold one position at a time, for one day.
            if prediction == 1: # Model predicts Bullish
                self.position.close() # Close any short position from a previous day
                self.buy() # Open a new long position
                
            elif prediction == 0: # Model predicts Bearish
                self.position.close() # Close any long position
                self.sell() # Open a new short position

Defining the ML-based trading strategy...


In [2]:
features_path = '../data/processed/hyp_a_features_advanced.parquet'
features = pd.read_parquet(features_path)
features

,day_of_week,asia_return,asia_range,london_direction,london_return,RSI_14,MOM_10,STOCHk_14_3_3,STOCHd_14_3_3,STOCHh_14_3_3,...,DEMA_10,MFI_14,BOP,ATRr_14,PSAR,MACD_12_26_9,MACDh_12_26_9,MACDs_12_26_9,TRIX_30_9,TRIXs_30_9
date,,,,,,,,,,,,,,,,,,,,,
2018-01-04,3,-0.002217,10.84,1,0.003565,45.078922,-6.11,22.171226,15.319033,6.852193,...,1307.859555,30.587346,0.779851,2.596249,1315.105707,-1.658247,-0.679946,-0.978301,-0.001497,-0.000003
2018-01-05,4,-0.003126,6.02,0,-0.003038,55.991011,1.03,62.873276,68.619618,-5.746342,...,1322.031673,77.862940,-0.853535,2.453258,1325.527730,2.172012,-0.035823,2.207835,0.005415,0.002398
2018-01-08,0,-0.002128,4.77,1,0.000759,45.015562,-2.39,59.471082,69.017323,-9.546241,...,1319.590210,65.308675,-0.721854,2.246994,1314.205989,0.393206,-0.220134,0.613340,0.008113,0.008157
2018-01-09,1,0.000212,4.95,0,-0.007216,51.761549,0.41,66.047107,63.291493,2.755614,...,1318.833379,61.996261,0.402878,1.891777,1315.422784,-0.006538,0.033848,-0.040386,0.003413,0.004123
2018-01-10,2,-0.003075,6.25,1,0.006139,36.431761,-3.03,28.646518,36.936254,-8.289736,...,1310.009542,46.961478,-0.506173,2.116085,1312.483658,-1.662555,-0.042516,-1.620039,-0.007136,-0.005239
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-10-06,0,0.012910,56.02,1,0.001191,77.465799,56.69,96.256108,92.696780,3.559328,...,3929.823490,64.533584,0.809028,12.148528,3905.487968,16.711563,5.015355,11.696208,0.014167,0.009970
2025-10-07,1,0.004472,21.58,1,0.001518,72.026423,18.11,87.543545,77.652362,9.891183,...,3973.098137,63.948098,0.469274,11.827424,3956.424691,15.175529,-0.842409,16.017938,0.043140,0.039891
2025-10-08,2,0.011485,48.33,1,0.006244,77.371131,45.83,98.762390,97.495532,1.266858,...,4016.236540,84.320633,0.964869,11.980454,3988.325533,15.659648,3.251609,12.408038,0.042414,0.042066


In [3]:
# --- Step 2: Define the Trading Strategy (FINAL CORRECTED Version) ---
print("Defining the ML-based trading strategy...")

class MLStrategy(Strategy):
    def init(self):
        print("Initializing strategy...")
        # model_path = '../models/xgb_classifier_hyp_a_xauusd_h1_2018_present.joblib'
        model_path = '../models/xgb_classifier_hyp_a_TUNED_paper_v3_final.joblib'
        self.model = joblib.load(model_path)
        print("Model loaded successfully.")

        features_path = '../data/processed/hyp_a_features_advanced.parquet'
        self.features = pd.read_parquet(features_path)
        print("Feature data loaded successfully.")
        
        self.london_tz = pytz.timezone('Europe/London')
        self._last_trade_date = None

    def next(self):
        current_time_utc = self.data.index[-1]
        current_date = current_time_utc.date()
        current_hour_utc = current_time_utc.hour
        
        if self._last_trade_date == current_date:
            return

        london_open_local = self.london_tz.localize(datetime.combine(current_date, time(8, 0)))
        london_open_utc = london_open_local.astimezone(pytz.utc)
        target_trade_hour_utc = london_open_utc.hour

        if current_hour_utc == target_trade_hour_utc:
            
            # --- THE FIX IS HERE ---
            # Convert the current_date object to a Pandas Timestamp to match the index type.
            current_date_ts = pd.to_datetime(current_date)
            
            # Now, check if this Timestamp is in our features index.
            if current_date_ts not in self.features.index:
                print(f"Skipping trade on {current_date}: No features found.")
                return

            self._last_trade_date = current_date
            
            # Use the Timestamp to locate the features
            today_features = self.features.loc[[current_date_ts]].drop(columns=['london_direction', 'london_return'])
            
            prediction = self.model.predict(today_features)[0]
            print(f"TRADE SIGNAL on {current_date}: Prediction is {'BULLISH' if prediction == 1 else 'BEARISH'}")
            
            if prediction == 1:
                self.position.close()
                self.buy()
            elif prediction == 0:
                self.position.close()
                self.sell()

Defining the ML-based trading strategy...


### **Step 3: Preparing the Data and Running the Backtest**
Now that our "brain" is defined, we need to load the historical price data, connect it to our strategy, and press "Go".


In [4]:
# --- Step 3: Load Data and Run the Backtest ---
print("\nPreparing data for backtest...")

# 1. Load the raw hourly price data. The backtester needs this to simulate trades.
price_data = pd.read_parquet('../data/raw/xauusd_h1_2018_present.parquet')
price_data.set_index('time', inplace=True)
price_data = price_data.tz_localize('UTC') # Make it timezone-aware

# --- THE FIX IS HERE ---
# The backtesting.py library requires specific column names with capital letters.
# Let's rename our columns to match its requirements.
price_data.rename(columns={
    'open': 'Open',
    'high': 'High',
    'low': 'Low',
    'close': 'Close',
    'tick_volume': 'Volume' # We'll rename 'tick_volume' to 'Volume'
}, inplace=True)
print("Price data columns renamed to match backtesting.py requirements.")
# --- END OF FIX ---

# 2. Isolate the test period. We must backtest ONLY on the data the model has NOT seen.
# This ensures the test is fair.
train_size_raw = int(len(price_data) * 0.8)
backtest_data = price_data.iloc[train_size_raw:]
print(f"Backtesting on data from {backtest_data.index[0]} to {backtest_data.index[-1]}")

# 3. Configure and initialize the backtest engine
bt = Backtest(
    backtest_data,     # The price data to run the simulation on
    MLStrategy,        # Our custom strategy "brain"
    cash=10000,        # Starting cash of $10,000
    commission=.0002,  # A 0.02% commission to simulate broker fees/spread
    exclusive_orders=True # Ensures one position at a time
)

# 4. Run the backtest!
print("\nRunning backtest...")
stats = bt.run()
trades_df = stats['_trades']
trades_df.to_csv('../data/processed/backtest_trades_paper.csv')
print("Backtest trades saved to 'data/processed/backtest_trades_paper.csv'")
print("Backtest complete.")

# 5. Print the results and generate the plot
print("\n--- Backtest Results ---")
print(stats)

print("\nGenerating equity curve plot...")
bt.plot()


Preparing data for backtest...
Price data columns renamed to match backtesting.py requirements.
Backtesting on data from 2024-03-25 05:00:00+00:00 to 2025-10-13 06:00:00+00:00

Running backtest...
Initializing strategy...
Model loaded successfully.
Feature data loaded successfully.


Backtest.run:   0%|          | 0/9179 [00:00<?, ?bar/s]

TRADE SIGNAL on 2024-03-25: Prediction is BULLISH
TRADE SIGNAL on 2024-03-26: Prediction is BULLISH
TRADE SIGNAL on 2024-03-27: Prediction is BULLISH
TRADE SIGNAL on 2024-03-28: Prediction is BULLISH
TRADE SIGNAL on 2024-04-01: Prediction is BULLISH
TRADE SIGNAL on 2024-04-02: Prediction is BULLISH
TRADE SIGNAL on 2024-04-03: Prediction is BULLISH
TRADE SIGNAL on 2024-04-04: Prediction is BULLISH
TRADE SIGNAL on 2024-04-05: Prediction is BULLISH
TRADE SIGNAL on 2024-04-08: Prediction is BULLISH
TRADE SIGNAL on 2024-04-09: Prediction is BULLISH
TRADE SIGNAL on 2024-04-10: Prediction is BULLISH
TRADE SIGNAL on 2024-04-11: Prediction is BULLISH
TRADE SIGNAL on 2024-04-12: Prediction is BULLISH
TRADE SIGNAL on 2024-04-15: Prediction is BULLISH
TRADE SIGNAL on 2024-04-16: Prediction is BULLISH
TRADE SIGNAL on 2024-04-17: Prediction is BULLISH
TRADE SIGNAL on 2024-04-18: Prediction is BULLISH
TRADE SIGNAL on 2024-04-19: Prediction is BULLISH
TRADE SIGNAL on 2024-04-22: Prediction is BULLISH


C:\Users\mecha\AppData\Local\Temp\ipykernel_17188\3018940854.py:39: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run()


Backtest trades saved to 'data/processed/backtest_trades_paper.csv'
Backtest complete.

--- Backtest Results ---
Start                     2024-03-25 05:00...
End                       2025-10-13 06:00...
Duration                    567 days 01:00:00
Exposure Time [%]                    99.72767
Equity Final [$]                  15099.38622
Equity Peak [$]                   15123.03278
Commissions [$]                    1699.46391
Return [%]                           50.99386
Buy & Hold Return [%]                86.67189
Return (Ann.) [%]                    29.28448
Volatility (Ann.) [%]                20.03431
CAGR [%]                             20.09685
Sharpe Ratio                          1.46172
Sortino Ratio                           2.848
Calmar Ratio                          2.67159
Alpha [%]                            -28.2515
Beta                                  0.91431
Max. Drawdown [%]                   -10.96144
Avg. Drawdown [%]                    -0.65691
Max. Drawdown

c:\Users\mecha\anaconda3\envs\trade_env\Lib\site-packages\bokeh\util\serialization.py:242: UserWarning: no explicit representation of timezones available for np.datetime64
  return convert(array.astype("datetime64[us]"))


GridPlot(id='p1336', ...)

In [13]:
pd.DataFrame(stats)

,0
Start,2024-03-08 22:00:00+00:00
End,2025-09-24 08:00:00+00:00
Duration,564 days 10:00:00
Exposure Time [%],99.879386
Equity Final [$],14088.627904
Equity Peak [$],14122.849708
Commissions [$],1654.481194
Return [%],40.886279
Buy & Hold Return [%],73.379048
Return (Ann.) [%],24.104429
